# Lesson 6: Advanced NER - NuNER & Model Comparisons

## 🎯 Learning Objectives

By the end of this lesson, you will:
1. Understand NuNER and its innovations over GLiNER
2. Compare different zero-shot NER approaches
3. Learn few-shot NER techniques
4. Master model selection for different scenarios
5. Build a comprehensive NER pipeline

---

## 📚 Table of Contents

1. [Introduction to NuNER](#1-introduction-to-nuner)
2. [NuNER vs GLiNER: Architecture Comparison](#2-nuner-vs-gliner-architecture-comparison)
3. [Using NuNER Models](#3-using-nuner-models)
4. [Few-Shot NER Techniques](#4-few-shot-ner-techniques)
5. [Comprehensive Model Comparison](#5-comprehensive-model-comparison)
6. [Model Selection Guide](#6-model-selection-guide)
7. [Building a Unified NER Pipeline](#7-building-a-unified-ner-pipeline)
8. [Further Reading](#8-further-reading)

---

## 📦 Setup & Installation

In [ ]:
# Install required packages
!pip install -q gliner torch transformers spacy datasets seqeval
!python -m spacy download en_core_web_sm -q

In [ ]:
# Import libraries
import torch
import time
import numpy as np
from gliner import GLiNER
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
import spacy
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")

---

## 1. Introduction to NuNER

### What is NuNER?

> **NuNER** (NuMind NER) is a family of compact NER models developed by NuMind AI. They're trained on LLM-annotated data and achieve state-of-the-art performance in zero-shot and few-shot NER tasks.
>
> — [NuNER Paper, 2024](https://arxiv.org/abs/2402.15343)

### NuNER Model Variants

| Model | Description | Best For |
|-------|-------------|----------|
| **NuNER Zero** | Token-level zero-shot NER | Long entities (>12 tokens) |
| **NuNER Zero-span** | Span-based prediction | Higher precision |
| **NuNER Zero-4k** | 4k context window | Long documents |

### Key Innovations

1. **LLM-Annotated Training Data**: Uses GPT-4 generated annotations for diversity
2. **Token Classification Approach**: Unlike GLiNER's span matching
3. **Better Long Entity Handling**: No 12-token limit
4. **MIT License**: Free for commercial use

### Performance Highlights

| Benchmark | NuNER Zero | GLiNER-large | Improvement |
|-----------|------------|--------------|-------------|
| Zero-shot F1 | 53.1% | 50.0% | +3.1% |
| Model Size | 110M | 340M | 3x smaller |
| Inference Speed | ~200ms | ~150ms | Comparable |

---

## 2. NuNER vs GLiNER: Architecture Comparison

### GLiNER Architecture

```
GLiNER (Span Matching):

Input: [ENT]person[ENT]org... + "Barack Obama visited Paris"
              │                            │
              └──────────┬─────────────────┘
                    BERT Encoder
                         │
              ┌──────────┴──────────┐
              │                     │
         Entity Emb           Span Emb
              │                     │
              └─────── Match ───────┘
                         │
                   Predictions
```

### NuNER Architecture

```
NuNER (Token Classification):

Input: entity_types + "Barack Obama visited Paris"
                           │
                     BERT Encoder
                           │
                  Token Representations
                           │
                  Token Classification
                           │
                 BIO Tags for each token
```

### Key Differences

| Aspect | GLiNER | NuNER |
|--------|--------|-------|
| **Approach** | Span matching | Token classification |
| **Max entity length** | ~12 tokens | Unlimited |
| **Speed** | Faster | Slightly slower |
| **Nested entities** | Supported | Limited |
| **Training data** | Manual annotations | LLM-generated |

---

## 3. Using NuNER Models

NuNER uses the GLiNER library but with NuMind models:

In [ ]:
# Load NuNER Zero model
# Note: NuNER uses the GLiNER architecture, so we load it with GLiNER
nuner = GLiNER.from_pretrained("numind/NuNER_Zero")

print("✅ NuNER Zero model loaded!")

In [ ]:
# Basic NuNER usage
text = """
Dr. Sarah Johnson, Chief Medical Officer at Massachusetts General Hospital, 
announced a breakthrough in Alzheimer's research. The study, published in 
Nature Medicine, shows promising results for early detection.
"""

labels = ["person", "job title", "organization", "disease", "publication"]

entities = nuner.predict_entities(text, labels, threshold=0.5)

print("🏷️ NuNER Extraction Results:\n")
print(f"Text: {text.strip()}\n")

for entity in entities:
    print(f"   {entity['text']:<40} → {entity['label']} ({entity['score']:.3f})")

In [ ]:
# Load GLiNER for comparison
gliner = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

print("✅ GLiNER model loaded!")

In [ ]:
# Side-by-side comparison
comparison_text = """
The World Health Organization declared COVID-19 a pandemic on March 11, 2020.
Dr. Tedros Adhanom Ghebreyesus, WHO Director-General, urged global cooperation.
"""

labels = ["organization", "disease", "date", "person", "title"]

print("📊 NuNER vs GLiNER Comparison\n")
print(f"Text: {comparison_text.strip()}\n")
print("=" * 70)

# NuNER
print("\n🔵 NuNER Results:")
nuner_entities = nuner.predict_entities(comparison_text, labels, threshold=0.5)
for e in nuner_entities:
    print(f"   {e['text']:<45} → {e['label']} ({e['score']:.3f})")

# GLiNER
print("\n🟢 GLiNER Results:")
gliner_entities = gliner.predict_entities(comparison_text, labels, threshold=0.5)
for e in gliner_entities:
    print(f"   {e['text']:<45} → {e['label']} ({e['score']:.3f})")

In [ ]:
# Test with long entities (NuNER's strength)
long_entity_text = """
The International Conference on Machine Learning and Artificial Intelligence 
Research Applications will be held in San Francisco next summer. The event 
features keynotes from leading researchers in natural language processing.
"""

labels = ["event", "location", "research field"]

print("📏 Long Entity Handling Test\n")
print(f"Text: {long_entity_text.strip()}\n")

# NuNER (better at long entities)
print("🔵 NuNER (no length limit):")
nuner_long = nuner.predict_entities(long_entity_text, labels, threshold=0.4)
for e in nuner_long:
    print(f"   [{len(e['text'].split())} words] {e['text']:<50} → {e['label']}")

# GLiNER
print("\n🟢 GLiNER (12 token limit):")
gliner_long = gliner.predict_entities(long_entity_text, labels, threshold=0.4)
for e in gliner_long:
    print(f"   [{len(e['text'].split())} words] {e['text']:<50} → {e['label']}")

---

## 4. Few-Shot NER Techniques

### What is Few-Shot NER?

Few-shot NER uses a small number of examples (typically 5-50) to improve extraction for specific entity types.

### Approaches to Few-Shot NER

1. **Prompt Engineering**: Provide examples in the prompt
2. **Fine-tuning**: Adapt model with few examples
3. **In-context Learning**: Use examples at inference time

In [ ]:
# Few-shot approach 1: Better label descriptions
# Detailed labels can act like "soft" few-shot learning

text = """
The Tesla Model S Plaid achieved a 0-60 mph time of 1.99 seconds.
BMW's M5 CS reaches 60 mph in 2.9 seconds.
"""

# Generic labels
generic_labels = ["product", "number"]

# Descriptive labels (few-shot via description)
descriptive_labels = ["car model name", "acceleration time in seconds"]

print("🏷️ Label Description as Few-Shot\n")
print(f"Text: {text.strip()}\n")

print("Generic labels:")
generic = nuner.predict_entities(text, generic_labels, threshold=0.4)
for e in generic:
    print(f"   {e['text']:<30} → {e['label']}")

print("\nDescriptive labels:")
descriptive = nuner.predict_entities(text, descriptive_labels, threshold=0.4)
for e in descriptive:
    print(f"   {e['text']:<30} → {e['label']}")

In [ ]:
# Few-shot approach 2: Ensemble with rule-based matching
import re

def few_shot_ner_ensemble(text, labels, model, patterns=None, threshold=0.5):
    """
    Combine zero-shot model with pattern-based rules.
    
    Args:
        text: Input text
        labels: Entity labels for zero-shot model
        model: GLiNER/NuNER model
        patterns: Dict of {label: [regex_patterns]}
        threshold: Confidence threshold
    """
    # Get model predictions
    model_entities = model.predict_entities(text, labels, threshold=threshold)
    
    # Add pattern-based entities
    if patterns:
        for label, pattern_list in patterns.items():
            for pattern in pattern_list:
                for match in re.finditer(pattern, text):
                    entity = {
                        'text': match.group(),
                        'label': label,
                        'start': match.start(),
                        'end': match.end(),
                        'score': 1.0,
                        'source': 'pattern'
                    }
                    
                    # Check for duplicates
                    is_duplicate = any(
                        e['start'] == entity['start'] and e['end'] == entity['end']
                        for e in model_entities
                    )
                    if not is_duplicate:
                        model_entities.append(entity)
    
    # Sort by position
    return sorted(model_entities, key=lambda x: x['start'])

# Example: Medical NER with patterns for drug dosages
medical_text = """
Patient prescribed Metformin 500mg twice daily and Lisinopril 10mg once daily.
Dr. Smith from Mayo Clinic recommended monitoring blood glucose levels.
"""

labels = ["medication", "doctor", "hospital"]

# Patterns for dosages (often missed by zero-shot)
patterns = {
    "dosage": [r"\d+\s*mg", r"\d+\s*ml"],
    "frequency": [r"(once|twice|three times)\s+daily", r"every\s+\d+\s+hours"]
}

print("🏥 Few-Shot Ensemble NER\n")
print(f"Text: {medical_text.strip()}\n")

# Zero-shot only
print("Zero-shot only:")
zero_shot = nuner.predict_entities(medical_text, labels, threshold=0.4)
for e in zero_shot:
    print(f"   {e['text']:<20} → {e['label']}")

# Ensemble
print("\nEnsemble (model + patterns):")
ensemble = few_shot_ner_ensemble(medical_text, labels, nuner, patterns, threshold=0.4)
for e in ensemble:
    source = e.get('source', 'model')
    print(f"   {e['text']:<20} → {e['label']:<15} [{source}]")

---

## 5. Comprehensive Model Comparison

Let's compare all the NER approaches we've learned:

In [ ]:
# Load all models
nlp_spacy = spacy.load("en_core_web_sm")
bert_ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

print("✅ All models loaded for comparison")

In [ ]:
# Comprehensive comparison
test_texts = [
    "Apple CEO Tim Cook announced the iPhone 15 at Apple Park in Cupertino.",
    "The European Central Bank raised interest rates by 0.25 percentage points.",
    "Dr. Fauci discussed COVID-19 vaccination at the National Institutes of Health."
]

def compare_all_models(text, gliner_model, nuner_model, spacy_nlp, bert_pipeline):
    """
    Compare NER results across all models.
    """
    results = {}
    
    # GLiNER
    gliner_labels = ["person", "organization", "location", "product"]
    start = time.time()
    gliner_ents = gliner_model.predict_entities(text, gliner_labels, threshold=0.5)
    results['GLiNER'] = {
        'entities': [(e['text'], e['label']) for e in gliner_ents],
        'time': time.time() - start
    }
    
    # NuNER
    start = time.time()
    nuner_ents = nuner_model.predict_entities(text, gliner_labels, threshold=0.5)
    results['NuNER'] = {
        'entities': [(e['text'], e['label']) for e in nuner_ents],
        'time': time.time() - start
    }
    
    # spaCy
    start = time.time()
    doc = spacy_nlp(text)
    results['spaCy'] = {
        'entities': [(ent.text, ent.label_) for ent in doc.ents],
        'time': time.time() - start
    }
    
    # BERT NER
    start = time.time()
    bert_ents = bert_pipeline(text)
    results['BERT'] = {
        'entities': [(e['word'], e['entity_group']) for e in bert_ents],
        'time': time.time() - start
    }
    
    return results

print("📊 Comprehensive Model Comparison\n")
print("=" * 80)

for text in test_texts:
    print(f"\nText: {text}\n")
    
    results = compare_all_models(text, gliner, nuner, nlp_spacy, bert_ner)
    
    print(f"{'Model':<12} {'Time':<10} {'Entities'}")
    print("-" * 70)
    
    for model_name, data in results.items():
        entities_str = str(data['entities'])[:60] + "..." if len(str(data['entities'])) > 60 else str(data['entities'])
        print(f"{model_name:<12} {data['time']*1000:.1f}ms{' '*4} {entities_str}")
    
    print()

In [ ]:
# Speed benchmark
import statistics

def benchmark_speed(text, model_func, n_runs=10):
    """Benchmark model inference speed."""
    times = []
    for _ in range(n_runs):
        start = time.time()
        model_func(text)
        times.append(time.time() - start)
    return {
        'mean': statistics.mean(times) * 1000,
        'std': statistics.stdev(times) * 1000 if len(times) > 1 else 0,
        'min': min(times) * 1000,
        'max': max(times) * 1000
    }

benchmark_text = "Google announced new AI features at their I/O conference in Mountain View."

benchmarks = {
    'spaCy': lambda t: nlp_spacy(t),
    'BERT NER': lambda t: bert_ner(t),
    'GLiNER': lambda t: gliner.predict_entities(t, ["company", "event", "location"], threshold=0.5),
    'NuNER': lambda t: nuner.predict_entities(t, ["company", "event", "location"], threshold=0.5),
}

print("⚡ Speed Benchmark (10 runs each)\n")
print(f"Text: {benchmark_text}\n")
print(f"{'Model':<12} {'Mean (ms)':<12} {'Std (ms)':<12} {'Min':<10} {'Max'}")
print("=" * 60)

for model_name, model_func in benchmarks.items():
    stats = benchmark_speed(benchmark_text, model_func)
    print(f"{model_name:<12} {stats['mean']:<12.2f} {stats['std']:<12.2f} {stats['min']:<10.2f} {stats['max']:.2f}")

---

## 6. Model Selection Guide

### Decision Framework

In [ ]:
# Interactive model selection guide
def recommend_model(requirements):
    """
    Recommend the best NER approach based on requirements.
    
    Args:
        requirements: dict with keys like:
            - custom_entities: bool
            - training_data: bool
            - speed_critical: bool
            - accuracy_critical: bool
            - long_entities: bool
    """
    recommendations = []
    
    if not requirements.get('custom_entities', False):
        recommendations.append({
            'model': 'spaCy / BERT NER',
            'reason': 'Standard entities (PER, ORG, LOC) - pre-trained models work great',
            'score': 90
        })
    
    if requirements.get('training_data', False):
        recommendations.append({
            'model': 'Fine-tuned BERT',
            'reason': 'Training data available - fine-tuning gives best accuracy',
            'score': 95 if requirements.get('accuracy_critical') else 85
        })
    
    if requirements.get('custom_entities', False) and not requirements.get('training_data', False):
        if requirements.get('long_entities', False):
            recommendations.append({
                'model': 'NuNER Zero',
                'reason': 'Custom entities + long entities - NuNER has no token limit',
                'score': 88
            })
        else:
            recommendations.append({
                'model': 'GLiNER',
                'reason': 'Custom entities, no training data - GLiNER is fast and accurate',
                'score': 85
            })
    
    if requirements.get('speed_critical', False):
        recommendations.append({
            'model': 'spaCy',
            'reason': 'Speed critical - spaCy is the fastest option',
            'score': 80
        })
    
    # Sort by score
    recommendations.sort(key=lambda x: x['score'], reverse=True)
    
    return recommendations

# Test scenarios
scenarios = [
    {
        'name': 'E-commerce product extraction',
        'requirements': {'custom_entities': True, 'training_data': False, 'speed_critical': True}
    },
    {
        'name': 'Medical NER with training data',
        'requirements': {'custom_entities': True, 'training_data': True, 'accuracy_critical': True}
    },
    {
        'name': 'Standard news NER',
        'requirements': {'custom_entities': False, 'speed_critical': True}
    },
    {
        'name': 'Legal document analysis',
        'requirements': {'custom_entities': True, 'long_entities': True, 'accuracy_critical': True}
    },
]

print("🎯 Model Selection Recommendations\n")
print("=" * 80)

for scenario in scenarios:
    print(f"\n📌 Scenario: {scenario['name']}")
    print(f"   Requirements: {scenario['requirements']}\n")
    
    recs = recommend_model(scenario['requirements'])
    for i, rec in enumerate(recs[:2]):
        print(f"   #{i+1} Recommendation: {rec['model']}")
        print(f"       Reason: {rec['reason']}")
        print(f"       Confidence: {rec['score']}%")

### Quick Reference Table

| Use Case | Best Model | Why |
|----------|------------|-----|
| General NER (PER, ORG, LOC) | spaCy / BERT NER | Fast, accurate for standard entities |
| Custom entities, no data | GLiNER | Zero-shot flexibility |
| Long custom entities | NuNER Zero | No token length limit |
| Maximum accuracy | Fine-tuned BERT | Learns domain patterns |
| Real-time processing | spaCy | Fastest inference |
| Nested entities | GLiNER (flat=False) | Supports overlapping spans |
| Low resource | GLiNER-small | Smallest model size |
| Multilingual | GLiNER-multi | Cross-lingual transfer |

---

## 7. Building a Unified NER Pipeline

Let's build a production-ready pipeline that combines multiple approaches:

In [ ]:
class UnifiedNERPipeline:
    """
    A unified NER pipeline that combines multiple approaches.
    
    Features:
    - Automatic model selection based on entity types
    - Fallback mechanisms
    - Confidence-based filtering
    - Entity deduplication
    """
    
    def __init__(self):
        # Standard entity models
        self.spacy_nlp = spacy.load("en_core_web_sm")
        self.bert_ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
        
        # Zero-shot models
        self.gliner = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")
        self.nuner = GLiNER.from_pretrained("numind/NuNER_Zero")
        
        # Standard entity type mapping
        self.standard_types = {
            'person': ['PERSON', 'PER'],
            'organization': ['ORG', 'ORGANIZATION'],
            'location': ['LOC', 'GPE', 'LOCATION'],
            'date': ['DATE'],
            'money': ['MONEY'],
        }
        
        print("✅ Unified NER Pipeline initialized")
    
    def _is_standard_type(self, entity_type):
        """Check if entity type is supported by standard models."""
        entity_type_lower = entity_type.lower()
        return entity_type_lower in self.standard_types
    
    def _normalize_label(self, label):
        """Normalize entity labels."""
        label_lower = label.lower()
        for normalized, variants in self.standard_types.items():
            if label_lower == normalized or label.upper() in variants:
                return normalized
        return label
    
    def extract(self, text, entity_types, threshold=0.5, use_ensemble=False):
        """
        Extract entities from text.
        
        Args:
            text: Input text
            entity_types: List of entity types to extract
            threshold: Confidence threshold for zero-shot models
            use_ensemble: Whether to combine multiple model outputs
        
        Returns:
            List of entity dictionaries
        """
        all_entities = []
        
        # Separate standard and custom entity types
        standard_types = [t for t in entity_types if self._is_standard_type(t)]
        custom_types = [t for t in entity_types if not self._is_standard_type(t)]
        
        # Extract standard entities with BERT/spaCy
        if standard_types:
            if use_ensemble:
                # Use both spaCy and BERT
                spacy_doc = self.spacy_nlp(text)
                for ent in spacy_doc.ents:
                    normalized_label = self._normalize_label(ent.label_)
                    if normalized_label in standard_types:
                        all_entities.append({
                            'text': ent.text,
                            'label': normalized_label,
                            'start': ent.start_char,
                            'end': ent.end_char,
                            'score': 0.9,
                            'source': 'spacy'
                        })
            
            bert_ents = self.bert_ner(text)
            for ent in bert_ents:
                normalized_label = self._normalize_label(ent['entity_group'])
                if normalized_label in standard_types:
                    all_entities.append({
                        'text': ent['word'],
                        'label': normalized_label,
                        'start': ent['start'],
                        'end': ent['end'],
                        'score': ent['score'],
                        'source': 'bert'
                    })
        
        # Extract custom entities with zero-shot models
        if custom_types:
            # Use NuNER for potentially long entities, GLiNER otherwise
            gliner_ents = self.gliner.predict_entities(text, custom_types, threshold=threshold)
            for ent in gliner_ents:
                all_entities.append({
                    'text': ent['text'],
                    'label': ent['label'],
                    'start': ent['start'],
                    'end': ent['end'],
                    'score': ent['score'],
                    'source': 'gliner'
                })
        
        # Deduplicate and filter
        unique_entities = self._deduplicate(all_entities, threshold)
        
        # Sort by position
        return sorted(unique_entities, key=lambda x: x['start'])
    
    def _deduplicate(self, entities, threshold):
        """Remove duplicate entities, keeping highest confidence."""
        seen = {}
        
        for ent in entities:
            key = (ent['start'], ent['end'], ent['label'])
            
            if key not in seen or ent['score'] > seen[key]['score']:
                seen[key] = ent
        
        return [e for e in seen.values() if e['score'] >= threshold]

# Initialize pipeline
unified_pipeline = UnifiedNERPipeline()

In [ ]:
# Test the unified pipeline
test_cases = [
    {
        'text': "Apple CEO Tim Cook announced the iPhone 15 Pro at Apple Park in Cupertino.",
        'types': ['person', 'organization', 'location', 'product']  # Mix of standard and custom
    },
    {
        'text': "Dr. Sarah Johnson prescribed Metformin 500mg for the patient's Type 2 Diabetes.",
        'types': ['person', 'medication', 'dosage', 'disease']  # Mostly custom
    },
    {
        'text': "The European Central Bank raised interest rates by 25 basis points on Thursday.",
        'types': ['organization', 'date', 'monetary policy term']  # Mix
    }
]

print("🔧 Unified NER Pipeline Results\n")
print("=" * 80)

for case in test_cases:
    print(f"\nText: {case['text']}")
    print(f"Entity types: {case['types']}\n")
    
    entities = unified_pipeline.extract(case['text'], case['types'], threshold=0.4)
    
    print(f"{'Entity':<30} {'Label':<20} {'Score':<8} {'Source'}")
    print("-" * 70)
    
    for ent in entities:
        print(f"{ent['text']:<30} {ent['label']:<20} {ent['score']:<8.3f} {ent['source']}")

---

## 8. Further Reading

### 📚 Research Papers

1. **NuNER: Entity Recognition Encoder Pre-training** (2024)
   - [arXiv:2402.15343](https://arxiv.org/abs/2402.15343)

2. **GLiNER: Generalist Model for Named Entity Recognition** (2023)
   - [arXiv:2311.08526](https://arxiv.org/abs/2311.08526)

3. **Few-Shot Named Entity Recognition** (2020)
   - [arXiv:2012.14978](https://arxiv.org/abs/2012.14978)

### 🔗 Resources

- [NuMind Models on Hugging Face](https://huggingface.co/numind)
- [GLiNER GitHub](https://github.com/urchade/GLiNER)
- [NuNER Collection](https://huggingface.co/collections/numind/nunerzero-zero-shot-ner-662b59803b9b438ff56e49e2)

---

## ✅ Lesson Summary

In this lesson, we covered:

1. **NuNER Architecture**: Token classification vs GLiNER's span matching
2. **Model Comparison**: NuNER vs GLiNER strengths and weaknesses
3. **Few-Shot Techniques**: Label engineering and ensemble methods
4. **Comprehensive Benchmarks**: Speed and accuracy comparisons
5. **Model Selection Guide**: Choosing the right approach
6. **Unified Pipeline**: Combining multiple models

### 🚀 Next Lesson Preview

In **Lesson 7**, we'll cover **NER Evaluation & Production Deployment**, including:
- Comprehensive evaluation strategies
- Error analysis techniques
- Production deployment patterns
- Monitoring and maintenance

In [ ]:
print("🎉 Congratulations! You've completed Lesson 6: Advanced NER")
print("\n📝 Key takeaways:")
print("   1. NuNER excels at long entities (no 12-token limit)")
print("   2. GLiNER is faster for shorter entities")
print("   3. Few-shot can be achieved through label engineering")
print("   4. Choose models based on entity types and requirements")
print("   5. Unified pipelines can leverage multiple models")
print("\n👉 Continue to Lesson 7: NER Evaluation & Production")